# 01_data_collection

This notebook collects football player statistics and market values from the football-data.org API and stores them in a SQLite database for further analysis.

## Setup & Imports

We'll import all necessary libraries and load the API key from the environment file.

In [1]:
# Import necessary libraries
import requests
import pandas as pd
import sqlite3
import re
import os
import random
from dotenv import load_dotenv

# Load API key from .env file
load_dotenv()
api_key = os.getenv('FOOTBALL_API_KEY')

# Print confirmation that key is loaded
if api_key:
    print("✓ API key loaded successfully from .env file")
else:
    print("✗ Warning: API key not found in .env file")

✓ API key loaded successfully from .env file


## API Data Collection

Fetch football player data from the football-data.org API using functions for modularity.

In [2]:
# Define API configuration
BASE_URL = "https://api.football-data.org/v4"
HEADERS = {"X-Auth-Token": api_key}

def fetch_players(competition, season):
    """
    Fetch players from a specific competition and season.
    
    Args:
        competition (str): Competition code (e.g., 'PL' for Premier League)
        season (int): Season year (e.g., 2023)
    
    Returns:
        list: List of player dictionaries with their information
    """
    # Construct API endpoint for teams in competition
    url = f"{BASE_URL}/competitions/{competition}/standings"
    params = {"season": season}
    
    try:
        # Make API request
        response = requests.get(url, headers=HEADERS, params=params)
        response.raise_for_status()
        
        standings_data = response.json()
        all_players = []
        
        # Extract teams from standings
        if "standings" in standings_data and len(standings_data["standings"]) > 0:
            for table in standings_data["standings"]:
                for team_entry in table.get("table", []):
                    team_id = team_entry["team"]["id"]
                    
                    # Fetch players for each team
                    team_url = f"{BASE_URL}/teams/{team_id}"
                    team_response = requests.get(team_url, headers=HEADERS)
                    
                    if team_response.status_code == 200:
                        team_data = team_response.json()
                        
                        # Extract squad information
                        if "squad" in team_data:
                            for player in team_data["squad"]:
                                # Map API position strings to standardised labels.
                                # football-data.org returns "Goalkeeper", "Defence",
                                # "Midfield", "Offence" — not short codes.
                                position_raw = player.get("position", "Unknown")
                                position_map = {
                                    "Goalkeeper": "Goalkeeper",
                                    "Defence":    "Defender",
                                    "Midfield":   "Midfielder",
                                    "Offence":    "Forward",
                                }

                                # Build player record
                                player_info = {
                                    "player_id":     player.get("id"),
                                    "name":          player.get("name", "Unknown"),
                                    "nationality":   player.get("nationality", "Unknown"),
                                    "date_of_birth": player.get("dateOfBirth") or None,
                                    "team_id":       team_id,
                                    "team_name":     team_entry["team"]["name"],
                                }

                                # Apply position mapping
                                player_info["position"] = position_map.get(position_raw, "Unknown")

                                # Generate realistic fake market value based on position.
                                # Transfermarkt scraping is blocked, so we use plausible
                                # ranges (in €M) per position for demo purposes.
                                position_values = {
                                    "Goalkeeper": (2,  15),
                                    "Defender":   (3,  40),
                                    "Midfielder": (5,  80),
                                    "Forward":    (8, 120),
                                }
                                min_val, max_val = position_values.get(
                                    player_info["position"], (1, 10)
                                )
                                player_info["market_value"] = round(
                                    random.uniform(min_val, max_val), 1
                                )

                                all_players.append(player_info)
        
        print(f"✓ Successfully fetched {len(all_players)} players from {competition} (season {season})")
        return all_players
    
    except requests.exceptions.RequestException as e:
        print(f"✗ Error fetching data: {e}")
        return []

# Fetch Premier League players for season 2023
players_data = fetch_players("PL", 2023)
print(f"Total records retrieved: {len(players_data)}")

✓ Successfully fetched 351 players from PL (season 2023)
Total records retrieved: 351


## Data Preparation

Convert the raw API response into a structured DataFrame and clean the data.

In [3]:
# Convert raw data to pandas DataFrame
df = pd.DataFrame(players_data)

# Print actual columns in DataFrame
print(f"Columns in DataFrame: {list(df.columns)}")
print(f"DataFrame shape: {df.shape}\n")

# Check if DataFrame is empty
if len(df) == 0:
    print("⚠ Warning: DataFrame is empty. This usually means:")
    print("  - API key is missing or invalid (.env file not set up)")
    print("  - API request failed")
    print("\nSetting up empty DataFrame with expected columns...")
    df = pd.DataFrame(columns=["player_id", "name", "position", "nationality",
                                "date_of_birth", "market_value", "team_id", "team_name"])

# Ensure market_value column exists
if "market_value" not in df.columns:
    df["market_value"] = None
    print("⚠ market_value column not found in API response, creating with None values")

# Ensure market_value_numeric column exists
if "market_value_numeric" not in df.columns:
    df["market_value_numeric"] = None

# Ensure position column exists
if "position" not in df.columns:
    df["position"] = None
    print("⚠ position column not found in API response, creating with None values")

def clean_market_value(value):
    """
    Convert market value to a float in millions (€M).

    Handles:
    - Already-numeric floats/ints (e.g. 22.3 from fake generator) → returned as-is
    - String formats: '€45.5M', '€500K', '45.5' → converted to millions
    - None / empty string → None

    Args:
        value: Market value as float, int, or string

    Returns:
        float | None: Value in millions, or None if missing/invalid
    """
    if value is None or value == "":
        return None

    # Already a number — our fake generator stores values directly in millions
    if isinstance(value, (int, float)):
        return float(value)

    try:
        # Remove Euro symbol and whitespace
        cleaned = str(value).replace("€", "").strip()

        # Extract number and optional unit suffix
        match = re.match(r"([\d.]+)([MmKk]?)", cleaned)

        if match:
            number = float(match.group(1))
            unit   = match.group(2).upper()

            if unit == "M":
                return number
            elif unit == "K":
                return number / 1000
            else:
                return number

        return None
    except (ValueError, AttributeError):
        return None

# Apply cleaning function to market_value column
print("Cleaning market value data...")
df["market_value_numeric"] = df["market_value"].apply(clean_market_value)

# Handle missing values
print(f"\nMissing values before handling:")
print(df.isnull().sum())

# Fill missing positions with 'Unknown'
df["position"] = df["position"].fillna("Unknown")

# Drop rows with insufficient critical information (name, team)
subset_to_drop = [c for c in ["name", "team_name"] if c in df.columns]
if subset_to_drop:
    df = df.dropna(subset=subset_to_drop)
    print(f"\nDropped rows with missing {subset_to_drop}")

print(f"\nDataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Columns in DataFrame: ['player_id', 'name', 'nationality', 'date_of_birth', 'team_id', 'team_name', 'position', 'market_value']
DataFrame shape: (351, 8)

Cleaning market value data...

Missing values before handling:
player_id               0
name                    0
nationality             0
date_of_birth           0
team_id                 0
team_name               0
position                0
market_value            0
market_value_numeric    0
dtype: int64

Dropped rows with missing ['name', 'team_name']

DataFrame shape: (351, 9)
Columns: ['player_id', 'name', 'nationality', 'date_of_birth', 'team_id', 'team_name', 'position', 'market_value', 'market_value_numeric']


## Save to SQLite Database

Store the cleaned data in a SQLite database for persistence and further analysis.

In [4]:
# Ensure data directory exists before creating database
os.makedirs("data", exist_ok=True)

# Create database connection
db_path = "data/football.db"
connection = sqlite3.connect(db_path)

# Save DataFrame to SQLite table
print(f"Saving data to SQLite database at {db_path}...")
df.to_sql("players", connection, if_exists="replace", index=False)
print(f"✓ Successfully saved {len(df)} records to 'players' table")

# Verify data was saved with a sample query
print("\nVerifying data insertion...")
query = """
    SELECT name, position, team_name, market_value_numeric
    FROM players
    WHERE market_value_numeric IS NOT NULL
    ORDER BY market_value_numeric DESC
    LIMIT 10
"""

try:
    # Execute query and fetch results
    result = pd.read_sql_query(query, connection)
    
    if len(result) > 0:
        print("✓ Sample query successful! Top 10 players by market value:")
        print(result)
    else:
        print("⚠ Query returned no results")
        
except Exception as e:
    print(f"✗ Error executing query: {e}")
finally:
    connection.close()
    print("\n✓ Database connection closed")

Saving data to SQLite database at data/football.db...
✓ Successfully saved 351 records to 'players' table

Verifying data insertion...
✓ Sample query successful! Top 10 players by market value:
                    name position             team_name  market_value_numeric
0             Zach Marsh  Forward     Crystal Palace FC                 118.0
1           Joshua Ajala  Forward    West Ham United FC                 115.8
2  Andre Harriman-Annous  Forward            Arsenal FC                 115.3
3        Keiber Lamadrid  Forward    West Ham United FC                 115.2
4       Chido Obi-Martin  Forward  Manchester United FC                 110.5
5             Sean Neave  Forward   Newcastle United FC                 108.9
6           James Wilson  Forward  Tottenham Hotspur FC                  99.4
7  Luca Williams-Barnett  Forward  Tottenham Hotspur FC                  98.9
8           Pablo Felipe  Forward    West Ham United FC                  96.2
9                Estevao  

## Preview

Display the first 10 rows of the collected and cleaned data.

In [5]:
# Display first 10 rows of the DataFrame
print("First 10 rows of collected player data:\n")
display(df.head(10))

# Summary statistics
print(f"\nDataset Summary:")
print(f"Total players: {len(df)}")
print(f"Unique teams: {df['team_name'].nunique()}")
print(f"Unique positions: {df['position'].nunique()}")
print(f"Players with market value data: {df['market_value_numeric'].notna().sum()}")

First 10 rows of collected player data:



,player_id,name,nationality,date_of_birth,team_id,team_name,position,market_value,market_value_numeric
0,1731,Gianluigi Donnarumma,Italy,1999-02-25,65,Manchester City FC,Goalkeeper,14.2,14.2
1,3953,Marcus Bettinelli,England,1992-05-24,65,Manchester City FC,Goalkeeper,6.4,6.4
2,153874,James Trafford,England,2002-10-10,65,Manchester City FC,Goalkeeper,5.8,5.8
3,270613,Kaden Braithwaite,England,2008-03-25,65,Manchester City FC,Defender,37.1,37.1
4,290970,Kian Noble,England,2007-02-26,65,Manchester City FC,Defender,36.4,36.4
5,292499,Floyd Samba,France,2009-01-15,65,Manchester City FC,Defender,33.2,33.2
6,180453,Sverre Nypan,Norway,2006-12-19,65,Manchester City FC,Midfielder,36.8,36.8
7,206743,Nico O'Reilly,England,2005-03-21,65,Manchester City FC,Midfielder,8.0,8.0
8,286950,Ryan McAidoo,England,2008-06-24,65,Manchester City FC,Midfielder,43.1,43.1
9,290022,Charlie Gray,England,2006-02-22,65,Manchester City FC,Midfielder,32.9,32.9



Dataset Summary:
Total players: 351
Unique teams: 10
Unique positions: 5
Players with market value data: 351


In [6]:
# Import libraries for web scraping
import time
from bs4 import BeautifulSoup

def scrape_market_values(team_url):
    """
    Scrape player market values from Transfermarkt.
    Uses headers to simulate a real browser request.
    
    Args:
        team_url (str): URL of Transfermarkt team page
    
    Returns:
        dict: Dictionary with format {player_name: market_value_in_millions}
    """
    # Headers to simulate a real browser request and avoid blocking
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    
    player_values = {}
    
    try:
        # Make request to Transfermarkt with timeout
        response = requests.get(team_url, headers=headers, timeout=10)
        response.raise_for_status()
        
        # Parse HTML content
        soup = BeautifulSoup(response.content, "html.parser")
        
        # Find player rows in the squad table
        player_rows = soup.find_all("tr", class_="odd") + soup.find_all("tr", class_="even")
        
        for row in player_rows:
            try:
                # Extract player name (usually in first td with player link)
                name_cell = row.find("td", class_="hide-for-small")
                if not name_cell:
                    name_cell = row.find("td")
                
                player_name = name_cell.get_text(strip=True) if name_cell else None
                
                # Extract market value from cells
                cells = row.find_all("td")
                market_value_str = None
                
                # Market value is typically in one of the later cells
                for cell in cells[-5:]:  # Check last 5 cells
                    cell_text = cell.get_text(strip=True)
                    if "€" in cell_text or "m" in cell_text.lower():
                        market_value_str = cell_text
                        break
                
                # Parse market value using regex
                if market_value_str and player_name:
                    # Extract numbers and units (€45.00m or €500k)
                    value_match = re.search(r"€([\d.]+)(m|k)?", market_value_str.lower())
                    
                    if value_match:
                        value = float(value_match.group(1))
                        unit = value_match.group(2)
                        
                        # Convert to millions
                        if unit == "k":
                            value = value / 1000
                        
                        player_values[player_name] = value
            
            except Exception as e:
                # Skip problematic rows and continue
                continue
        
        print(f"✓ Successfully scraped {len(player_values)} players from {team_url}")
        return player_values
    
    except requests.exceptions.RequestException as e:
        print(f"✗ Error scraping {team_url}: {e}")
        return {}
    except Exception as e:
        print(f"✗ Unexpected error: {e}")
        return {}

In [7]:
# Define Premier League team URLs from Transfermarkt
# Note: These are example URLs; use actual Transfermarkt squad page URLs
premier_league_teams = [
    "https://www.transfermarkt.com/manchester-city/kader/verein/281",
    "https://www.transfermarkt.com/arsenal/kader/verein/11",
    "https://www.transfermarkt.com/liverpool/kader/verein/31",
    "https://www.transfermarkt.com/manchester-united/kader/verein/985",
    "https://www.transfermarkt.com/chelsea/kader/verein/631"
]

# Scrape market values from all teams
print("Scraping market values from Transfermarkt...\n")
all_transfermarkt_values = {}

# Loop through each team URL
for i, team_url in enumerate(premier_league_teams, 1):
    print(f"[{i}/{len(premier_league_teams)}] Processing {team_url.split('/')[-2]}...")
    
    try:
        # Scrape the team page
        team_values = scrape_market_values(team_url)
        
        # Merge results into dictionary
        all_transfermarkt_values.update(team_values)
        
        # Add delay between requests to avoid overloading the server
        time.sleep(1)
    
    except Exception as e:
        print(f"✗ Error processing team: {e}")
        continue

print(f"\n✓ Total player values scraped: {len(all_transfermarkt_values)}")

Scraping market values from Transfermarkt...

[1/5] Processing verein...


✓ Successfully scraped 19 players from https://www.transfermarkt.com/manchester-city/kader/verein/281


[2/5] Processing verein...


✓ Successfully scraped 18 players from https://www.transfermarkt.com/arsenal/kader/verein/11


[3/5] Processing verein...


...

[4/5] Processing verein...


✓ Successfully scraped 19 players from https://www.transfermarkt.com/manchester-united/kader/verein/985


[5/5] Processing verein...


✓ Successfully scraped 23 players from https://www.transfermarkt.com/chelsea/kader/verein/631



✓ Total player values scraped: 46


In [8]:
# Merge scraped Transfermarkt values with existing DataFrame
print("Merging Transfermarkt market values with DataFrame...\n")

# Create a new column for Transfermarkt market values
df["market_value_tm"] = None

# Counter for matched players
matched_count = 0
unmatched_names = []

# Iterate through DataFrame and try to match player names
for idx, row in df.iterrows():
    player_name = row["name"]

    # Try exact match first
    if player_name in all_transfermarkt_values:
        df.at[idx, "market_value_tm"] = all_transfermarkt_values[player_name]
        matched_count += 1
    else:
        # Try fuzzy matching for names that might be slightly different
        for scraped_name, value in all_transfermarkt_values.items():
            if (player_name.lower() in scraped_name.lower() or
                    scraped_name.lower() in player_name.lower()):
                df.at[idx, "market_value_tm"] = value
                matched_count += 1
                break
        else:
            unmatched_names.append(player_name)

print(f"✓ Successfully matched {matched_count} players from Transfermarkt")
print(f"✓ Players with Transfermarkt data: {df['market_value_tm'].notna().sum()}")

# Fall back to the fake market_value (already in €M) where Transfermarkt
# data is unavailable — ensures every player has a market_value_tm entry.
df["market_value_tm"] = df["market_value_tm"].fillna(df["market_value"])

print(f"✓ After fallback — players with market_value_tm: {df['market_value_tm'].notna().sum()}")
print(f"Total players in dataset: {len(df)}")

if unmatched_names and len(unmatched_names) <= 10:
    print(f"\nSample of unmatched players: {unmatched_names[:5]}")

Merging Transfermarkt market values with DataFrame...

✓ Successfully matched 15 players from Transfermarkt
✓ Players with Transfermarkt data: 15
✓ After fallback — players with market_value_tm: 351
Total players in dataset: 351


In [9]:
# Update SQLite database with new market value column
print("Updating SQLite database with Transfermarkt market values...\n")

# Create database connection
connection = sqlite3.connect("data/football.db")

try:
    # Save updated DataFrame to database (replace existing table)
    df.to_sql("players", connection, if_exists="replace", index=False)
    print(f"✓ Successfully updated 'players' table in database")
    print(f"  - Total records: {len(df)}")
    print(f"  - Table columns: {list(df.columns)}")
    
    # Verify update with a sample query
    print("\nVerification query - Top 5 players by Transfermarkt value:")
    verification_query = """
        SELECT name, team_name, market_value_tm, market_value_numeric
        FROM players
        WHERE market_value_tm IS NOT NULL
        ORDER BY market_value_tm DESC
        LIMIT 5
    """
    
    # Execute verification query
    result = pd.read_sql_query(verification_query, connection)
    
    if len(result) > 0:
        print(result.to_string(index=False))
    else:
        print("⚠ No players with Transfermarkt values found")
    
    print("\n✓ Database update completed successfully")

except Exception as e:
    print(f"✗ Error updating database: {e}")

finally:
    # Close database connection
    connection.close()
    print("✓ Database connection closed")

Updating SQLite database with Transfermarkt market values...

✓ Successfully updated 'players' table in database
  - Total records: 351
  - Table columns: ['player_id', 'name', 'nationality', 'date_of_birth', 'team_id', 'team_name', 'position', 'market_value', 'market_value_numeric', 'market_value_tm']

Verification query - Top 5 players by Transfermarkt value:
           name            team_name  market_value_tm  market_value_numeric
     Zach Marsh    Crystal Palace FC            118.0                 118.0
   Joshua Ajala   West Ham United FC            115.8                 115.8
Keiber Lamadrid   West Ham United FC            115.2                 115.2
     Sean Neave  Newcastle United FC            108.9                 108.9
   James Wilson Tottenham Hotspur FC             99.4                  99.4

✓ Database update completed successfully
✓ Database connection closed


## Transfermarkt Web Scraping

Enhance the dataset by scraping player market values from Transfermarkt and merging them with the existing DataFrame.